# Level 1 Task 3: Data Cleaning
**Oasis Infobyte Data Analytics Internship**

### Project Overview
This notebook demonstrates foundational data cleaning techniques on a messy transactional dataset.

### Steps Performed:
1. **Standardized Missing Values:** Converted placeholder text like `"ERROR"` and `"UNKNOWN"` into recognizable `NaN` values.
2. **Type Conversion:** Casted string representations of numbers and dates into proper `float64` and `datetime64` formats for analysis.
3. **Data Imputation:** Mathematically calculated missing `Total Spent` values by multiplying `Quantity` by `Price Per Unit`.
4. **Categorical Handling:** Filled unrecoverable categorical data (`Payment Method`, `Location`) with an `'Unspecified'` label to maintain dataset integrity.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('dirty_cafe_sales.csv')

# 1. Basic Information
print("--- Dataset Info ---")
print(df.info())

# 2. Check for Missing (Null) Values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 3. Check for Duplicate Rows
print(f"\n--- Total Duplicate Rows: {df.duplicated().sum()} ---")

# 4. Preview the Data
display(df.head())

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 1.1 MB
None

--- Missing Values Count ---
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

--- Total Duplicate Rows: 0 ---


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# 1. Replace fake missing values ("ERROR", "UNKNOWN") with actual pandas NaN
df.replace(['ERROR', 'UNKNOWN', 'Unknown'], np.nan, inplace=True)

# 2. Convert text columns to their correct numeric and datetime formats
# (errors='coerce' will force any other weird text we missed into NaN)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

# 3. Smart Imputation: Calculate missing 'Total Spent' where possible
# If Total Spent is NaN, but we have Quantity and Price, we can just multiply them!
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

# 4. Handle categorical missing values
df['Payment Method'] = df['Payment Method'].fillna('Unspecified')
df['Location'] = df['Location'].fillna('Unspecified')

# 5. Drop rows where critical core information is completely missing
df.dropna(subset=['Item', 'Transaction Date', 'Quantity', 'Price Per Unit'], inplace=True)

# 6. Check our beautifully cleaned dataset!
print("--- Cleaned Missing Values Count ---")
print(df.isnull().sum())
print("\n--- Cleaned Data Types ---")
print(df.dtypes)
display(df.head())

--- Cleaned Missing Values Count ---
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

--- Cleaned Data Types ---
Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unspecified,Unspecified,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
# Save the cleaned dataset to a new CSV file
df.to_csv('cleaned_dataset.csv', index=False)
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
